# 질감 마스크 '구멍' 의존 확인 — 이미 학습된 ConvNeXt로 추론만 (학습 없음)

**왜 하나**
- 질감 마스크는 언더필 틈을 회색 구멍으로 지운다. 공통 3,039장에서 구멍 10% 이상인 사진은 정상 1장, 언더필 187장이고, 구멍 비율 하나만으로 AUROC 0.851 [확인].
- 질감 마스크로 학습한 모델이 결함 자체가 아니라 **이 구멍(마스크 흔적)** 에 기대는지 직접 확인한다.

| 시험 | 입력 변형 | 구멍에 기대는 모델이라면 |
|---|---|---|
| A. 구멍 메우기 | 언더필 사진의 구멍(출력물 안쪽에 갇힌 회색)만 원본 화소로 되돌림 | 언더필 확률 ↓, 미탐 ↑ |
| B. 구멍 옮겨 심기 | 정상 사진에 언더필 사진의 구멍 모양을 회색으로 칠함 (결함 없음) | 정상인데 언더필 판정 ↑ (오탐 ↑) |
| 대조군 | 같은 시험을 재활용 × 재활용 모델(구멍 없는 입력으로 학습)에도 적용 | 대조군도 똑같이 반응하면 "학습된 흔적"이 아니라 구멍이 원래 결함처럼 보이는 것 |

- 모델: 질감 × 질감(10/05), 2단계 × 질감(10/13), 재활용 × 재활용(10/05), 각 3시드 × 5fold. 각 사진은 OOF였던 fold 모델로만 채점.
- **재현 확인:** 바꾸지 않은 입력의 점수가 저장된 OOF 점수와 같아야 한다 (기본 허용 0.02). 다르면 멈춘다 — 입력 처리가 학습 때와 다르다는 뜻이라 A/B를 믿을 수 없음.
- 구멍 정의: 질감 마스크가 지운 곳 중 재활용 마스크가 출력물로 보는 곳이면서, 재활용 마스크 바깥 회색과 이어지지 않은 덩어리.
- 옮겨 심을 구멍: 구멍 ≥ 2%인 언더필 사진의 구멍 모양을 프레임 순서대로 정상 사진에 하나씩 대응 (정상 사진의 출력물 영역 안에만 칠함).

**Drive `산공종설/`에 필요한 것:** 아래 4개 zip은 이미 있음 + 학습 결과 폴더 2개(가중치 포함)도 이미 있음. **새로 올릴 것: `hole_probe_bundle_20261016.zip`**
- `underfill_originals_4000.zip`, `dinov2_grid_bundle_20261013.zip`, `convnext_three_preproc_patch_20261005.zip`, `resnet18_hh_straight_shape_rescue_bundle_20261005_area_gt10_v3.zip`
- `convnext_three_preproc_20261005_run01/` (v2, v3_texture 가중치), `convnext_stage2v32_20261013_run01/` (v4_s2_texture 가중치)

**런타임:** GPU 아무거나 (T4/L4 충분). 시간 대부분이 입력 준비(CPU, 30~40분 [추정])라 비싼 GPU는 낭비. 추론은 수 분.

## 1. 설정 + GPU·Drive

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
OLD_RUN = "convnext_three_preproc_20261005_run01"      # v2(재활용×재활용), v3_texture(질감×질감)
S2_RUN = "convnext_stage2v32_20261013_run01"           # v4_s2_texture(2단계×질감)
SEEDS = [0, 1, 2]
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "GPU 런타임이 아님"
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os, glob, shutil
RESULTS = f"{DRIVE_DIR}/results/hole_probe_20261016"; os.makedirs(RESULTS, exist_ok=True)
for run, ver in ((OLD_RUN, "v3_texture"), (OLD_RUN, "v2"), (S2_RUN, "v4_s2_texture")):
    n = len(glob.glob(f"{DRIVE_DIR}/{run}/{ver}/seed*/fold*/model_state.pt"))
    print(f"{run}/{ver}: 가중치 {n}개 (15개여야 함)")
print("OOF 파일:", [os.path.exists(f"{DRIVE_DIR}/{r}/oof_predictions.csv") for r in (OLD_RUN, S2_RUN)])

## 2. 압축 풀기

In [ ]:
ZIPS = {"underfill_originals_4000.zip": "/content/orig", "dinov2_grid_bundle_20261013.zip": "/content/grid",
        "convnext_three_preproc_patch_20261005.zip": "/content/texpatch",
        "resnet18_hh_straight_shape_rescue_bundle_20261005_area_gt10_v3.zip": "/content/rbundle",
        "hole_probe_bundle_20261016.zip": "/content/probe"}
for z, d in ZIPS.items():
    assert os.path.exists(f"{DRIVE_DIR}/{z}"), f"없음: {z}"
    shutil.copy2(f"{DRIVE_DIR}/{z}", "/content/")
    !unzip -q -o "/content/{z}" -d "{d}"
!find /content/orig -name "*.jpg" | wc -l
!python /content/probe/hole_probe.py --self-test

## 3. 실행 (입력 준비 → 추론 → 요약)
로그는 Drive `results/hole_probe_20261016/probe_log.txt`. 화면에는 진행 줄만 나온다. 끊기면 다시 실행: 입력 캐시(`/content/probe_cache`)가 남아 있으면 준비를 건너뛴다.

In [ ]:
import subprocess, sys
cmd = [sys.executable, "-u", "/content/probe/hole_probe.py", "--grid", "/content/grid", "--texture", "/content/texpatch/texture",
       "--bundle", "/content/rbundle", "--originals", "/content/orig", "--old-run", f"{DRIVE_DIR}/{OLD_RUN}",
       "--s2-run", f"{DRIVE_DIR}/{S2_RUN}", "--cache", "/content/probe_cache", "--output", RESULTS, "--seeds", *map(str, SEEDS)]
with open(f"{RESULTS}/probe_log.txt", "a", encoding="utf-8") as log:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        log.write(line); log.flush()
        if any(k in line for k in ("구멍 계산", "입력", "완료", "reproduction", "실패", "Error", "Traceback")): print(line.rstrip())
    print("종료 코드", p.wait())

## 4. 결과
- **repro_check:** `max_abs_diff`가 모두 0.02 이하여야 아래 결과를 믿을 수 있다.
- **A (구멍 메우기, 언더필 사진):** 질감 모델에서 `메웠더니_정상으로바뀜`이 많고 대조군에서 적으면 → 질감 모델이 구멍에 기댄다는 증거.
- **B (구멍 옮겨 심기, 정상 사진):** 질감 모델에서 `추가했더니_언더필로바뀜`이 많고 대조군에서 적으면 → 같은 결론. B가 더 강한 증거 (결함 없이 구멍만으로 판정이 바뀜).
- 대조군은 원래 구멍 없는 입력으로 학습했으므로, 대조군 A의 '구멍있음' 입력은 낯선 입력이다 (참고용).

**이 셀 출력을 붙여 주거나 `results/hole_probe_20261016.zip`을 주면 된다.**

In [ ]:
import pandas as pd, json
print(json.load(open(f"{RESULTS}/run_info.json", encoding="utf-8")))
for f in ("repro_check.csv", "A_fill_holes.csv", "B_add_holes.csv"):
    if os.path.exists(f"{RESULTS}/{f}"):
        print("\n==", f); print(pd.read_csv(f"{RESULTS}/{f}").to_string(index=False))
print(shutil.make_archive(f"{DRIVE_DIR}/results/hole_probe_20261016", "zip", RESULTS))

## 5. 런타임 반납

In [ ]:
from google.colab import runtime
drive.flush_and_unmount(); runtime.unassign()